# Vivir más sin ser más rico

**Trabajo Práctico Final · Visualización de la Información · ITBA**  
Javier Peral Belmont y Pilar Frutos · Noviembre de 2026

En 2024, Costa Rica vivía 2,1 años más que Estados Unidos con poco más de un tercio de su PIB por habitante. ¿Qué tan bien describe el ingreso las diferencias de longevidad entre países? Primero miramos el patrón y sus excepciones. Después comparamos la esperanza de vida en países con distintos niveles de acceso al agua y de vacunación.

**En Colab**, elegí *Entorno de ejecución → Ejecutar todo* para descargar los datos de Data360 y generar los gráficos.

In [308]:
!pip -q install babel pycountry

import time

import altair as alt
import numpy as np
import pandas as pd
import pycountry
import requests
from babel import Locale

alt.data_transformers.disable_max_rows()
# Números con coma decimal y punto de miles en ejes y fichas
alt.renderers.set_embed_options(format_locale={"decimal": ",", "thousands": ".", "grouping": [3], "currency": ["$", ""]})

API = "https://data360api.worldbank.org/data360"
DESDE, HASTA = 2000, 2024


In [309]:
def pedir(metodo, ruta, intentos=6, **kwargs):
    """Llama a la API reintentando: a veces responde con el cuerpo vacío o corta la conexión."""
    for intento in range(intentos):
        try:
            r = requests.request(metodo, f"{API}/{ruta}", timeout=120, **kwargs)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))


# 1. Exploración de Data360

Consultamos el catálogo de Data360 del Banco Mundial y contamos los indicadores por base. Elegimos **World Development Indicators (WDI)** porque reúne series anuales por país de economía, población y salud.

In [310]:
respuesta = pedir("POST", "searchv2", json={
    "search": "*",
    "top": 0,
    "count": True,
    "filter": "type eq 'indicator'",
    "facets": ["series_description/database_id,count:1000"],
})

bases = (pd.DataFrame(respuesta["@search.facets"]["series_description/database_id"])
         .rename(columns={"value": "base", "count": "indicadores"})
         .sort_values("indicadores", ascending=False, ignore_index=True))

print("Bases de datos en Data360:", len(bases))
print("Indicadores en total:", bases["indicadores"].sum())
bases.head(15)


Bases de datos en Data360: 171
Indicadores en total: 9850


,base,indicadores
0,WB_WDI,1545
1,IMF_BOP,1146
2,WB_EDSTATS,1072
3,WB_ES,584
4,IMF_IFS,495
5,WB_GS,363
6,WB_FINDEX,280
7,IMF_GFS_SOO,218
8,BS_SGI,193
9,WB_HNP,192


# 2. Selección de indicadores y preparación de los datos

| Indicador de WDI | Código en Data360 | Unidad |
|---|---|---|
| Esperanza de vida al nacer | `WB_WDI_SP_DYN_LE00_IN` | Años |
| PIB por habitante (PPA) | `WB_WDI_NY_GDP_PCAP_PP_KD` | Dólares internacionales constantes de 2021 |
| Acceso básico a agua para beber | `WB_WDI_SH_H2O_BASW_ZS` | Porcentaje de la población |
| Vacunación contra el sarampión (primera dosis) | `WB_WDI_SH_IMM_MEAS` | Porcentaje de niños de 12 a 23 meses |
| Finalización del primer ciclo de secundaria | `WB_WDI_SE_SEC_CMPT_LO_ZS` | Tasa bruta, como porcentaje de la población en edad de terminar el ciclo |

Evaluamos secundaria como candidata para el tercer gráfico. El acceso básico al agua se refiere al uso de una fuente mejorada a no más de 30 minutos de ida y vuelta.

### Cómo llega una observación

Este ejemplo muestra el registro de esperanza de vida de Argentina en 2024. La API identifica la economía con `REF_AREA`, el año con `TIME_PERIOD` y el valor con `OBS_VALUE`. Usamos los totales cuando vienen desagregados y cruzamos las series por economía y año.

In [315]:
obs = pedir("GET", "data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": "WB_WDI_SP_DYN_LE00_IN",
                                   "REF_AREA": "ARG", "TIME_PERIOD": 2024})["value"][0]
pd.DataFrame([obs]).T


,0
OBS_VALUE,77.543
TIME_FORMAT,P1Y
UNIT_MULT,0
COMMENT_OBS,None
OBS_STATUS,A
OBS_CONF,PU
AGG_METHOD,_Z
DECIMALS,2
COMMENT_TS,None
DATA_SOURCE,None


In [317]:
def traer_indicador(codigo, columna):
    """Descarga todas las páginas de un indicador y devuelve una tabla país–año."""
    filas, skip = [], 0
    while True:
        pagina = pedir("GET", "data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": codigo, "top": 1000, "skip": skip})
        filas += pagina["value"]
        skip += 1000
        if skip >= pagina["count"] or not pagina["value"]:
            break
    d = pd.DataFrame(filas)
    for dim in ["SEX", "AGE", "URBANISATION"]:      # nos quedamos con el total si hay desagregaciones
        if dim in d and d[dim].nunique() > 1:
            d = d[d[dim] == "_T"]
    return pd.DataFrame({
        "iso3": d["REF_AREA"],
        "anio": pd.to_numeric(d["TIME_PERIOD"]).astype(int),
        columna: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    }).dropna()


INDICADORES = {
    "vida": "WB_WDI_SP_DYN_LE00_IN",
    "pib": "WB_WDI_NY_GDP_PCAP_PP_KD",
    "agua": "WB_WDI_SH_H2O_BASW_ZS",
    "sarampion": "WB_WDI_SH_IMM_MEAS",
    "secundaria": "WB_WDI_SE_SEC_CMPT_LO_ZS",
}

series = {clave: traer_indicador(codigo, clave) for clave, codigo in INDICADORES.items()}

La API incluye agregados como el total mundial. Los excluimos con la lista de economías del Banco Mundial, que también aporta la región y el grupo de ingreso de cada país.

In [318]:
jerarquia = pedir("GET", "portal/v1/hierarchy", params={"type": "country"})["data"]

REGIONES = {
    "East Asia & Pacific": "Asia oriental y Pacífico",
    "Europe & Central Asia": "Europa y Asia central",
    "Latin America & Caribbean": "América Latina y el Caribe",
    "Middle East, North Africa, Afghanistan & Pakistan": "Medio Oriente y norte de África",
    "North America": "América del Norte",
    "South Asia": "Asia meridional",
    "Eastern and Southern Africa": "África subsahariana",
    "Western and Central Africa": "África subsahariana",
}
INGRESOS = {"Low income": "Bajo", "Lower middle income": "Medio bajo", "Upper middle income": "Medio alto", "High income": "Alto"}

regiones = pd.DataFrame([{"iso3": c["id"], "nombre_en": c["name"], "region": REGIONES[g["name"]]}
                         for g in jerarquia["HCL_WB_REGIONS"] for c in g["countries"]])
ingresos = pd.DataFrame([{"iso3": c["id"], "ingreso": INGRESOS.get(g["name"])}
                         for g in jerarquia["HCL_WB_INCOME"] for c in g["countries"]])
paises = regiones.merge(ingresos, on="iso3", how="left")
paises["ingreso"] = paises["ingreso"].fillna("Sin clasificar")

nombres_es = Locale("es").territories
def nombre_es(fila):
    pais = pycountry.countries.get(alpha_3=fila.iso3)
    iso2 = pais.alpha_2 if pais else {"XKX": "XK"}.get(fila.iso3)
    return nombres_es.get(iso2, {"CHI": "Islas del Canal"}.get(fila.iso3, fila.nombre_en))

paises["pais"] = paises.apply(nombre_es, axis=1)

In [319]:
panel = None
for s in series.values():
    panel = s if panel is None else panel.merge(s, on=["iso3", "anio"], how="outer")

panel = (panel[panel["anio"].between(1990, HASTA)]
         .merge(paises[["iso3", "pais", "region", "ingreso"]], on="iso3", how="inner")
         .sort_values(["iso3", "anio"], ignore_index=True))

**Período y cobertura.** Usamos 2024 para el PIB y 2000–2024 para agua y vacunación. El agua tiene datos desde 2000 y, en la descarga, la esperanza de vida llega hasta 2024. Cada gráfico muestra solo las economías con los datos que necesita.

**Por qué elegimos vacunación.** Solo 62 economías tenían datos de secundaria y esperanza de vida en al menos 20 de los 25 años. La vacunación tenía entre 190 y 193 economías con ambos datos cada año, lo que daba una base más estable para las comparaciones.

In [322]:
periodo = panel[panel["anio"].between(DESDE, HASTA)].dropna(subset=["vida"])
anios_secundaria = periodo.dropna(subset=["secundaria"]).groupby("iso3")["anio"].nunique()
paises_vacunacion = periodo.dropna(subset=["sarampion"]).groupby("anio")["iso3"].nunique()

print(f"Secundaria: {(anios_secundaria >= 20).sum()} economías con datos en al menos 20 de los 25 años.")
print(f"Vacunación: entre {paises_vacunacion.min()} y {paises_vacunacion.max()} economías con datos por año.")

Secundaria: 62 economías con datos en al menos 20 de los 25 años.
Vacunación: entre 190 y 193 economías con datos por año.


# 3. ¿Los países con mayor ingreso viven más?

La línea estima la esperanza de vida según el PIB por habitante. Cada duplicación del ingreso ocupa la misma distancia horizontal. El color marca la **brecha** entre la esperanza de vida observada y la estimada. Azul y rojo indican una diferencia de al menos 1,5 años por encima o por debajo. Amarillo indica que el país queda cerca de la línea.

Usá **Comparar**, **Región** o **Buscar país** para explorar la dispersión. Pasá el cursor sobre un punto para ver los valores.

Ajustamos una regresión lineal de la esperanza de vida al nacer sobre el logaritmo del PIB por habitante.

In [329]:
corte = panel[panel["anio"] == 2024].dropna(subset=["vida", "pib"]).copy()
b, a = np.polyfit(np.log(corte["pib"]), corte["vida"], 1)
corte["esperada"] = a + b * np.log(corte["pib"])
corte["brecha"] = corte["vida"] - corte["esperada"]
r2 = 1 - corte["brecha"].var() / corte["vida"].var()

In [328]:
import numpy as np

columnas = ['pais', 'region', 'pib', 'vida', 'esperada', 'brecha']
g1 = corte.copy()
g1 = g1.dropna(subset=columnas)
g1 = g1[g1['pib'] > 0].copy()

def categoria(brecha):
    if brecha >= 1.5:
        return 'Por encima'
    if brecha <= -1.5:
        return 'Por debajo'
    return 'Cerca del modelo'

g1['comparacion'] = g1['brecha'].apply(categoria)
g1['destacado'] = g1['pais'].isin([
    'Japón', 'Costa Rica', 'Estados Unidos',
    'Argentina', 'India', 'Nigeria'
])

curva = pd.DataFrame({'pib': np.geomspace(700, 150_000, 120)})
curva['vida'] = a + b * np.log(curva['pib'])

vista = alt.param(
    name='vista', value='Todos',
    bind=alt.binding_radio(
        options=['Todos', 'Por encima', 'Cerca del modelo', 'Por debajo'],
        name='Comparar: '
    )
)
region = alt.param(
    name='region', value='Todas',
    bind=alt.binding_select(
        options=['Todas'] + sorted(g1['region'].dropna().unique().tolist()),
        name='Región: '
    )
)
buscar = alt.param(
    name='buscar', value='',
    bind=alt.binding(input='search', placeholder='Ej.: Chile', name='Buscar país: ')
)

filtro = "(vista == 'Todos' || datum.comparacion == vista) && (region == 'Todas' || datum.region == region)"
coincide = "indexof(lower(datum.pais), lower(trim(buscar))) >= 0"
hay_busqueda = "length(trim(buscar)) > 0"

x = alt.X(
    'pib:Q', title='PIB por habitante (miles de dólares)',
    scale=alt.Scale(type='log', domain=[700, 150_000]),
    axis=alt.Axis(
        values=[1000, 2000, 5000, 10_000, 20_000, 50_000, 100_000],
        labelExpr="format(datum.value / 1000, '.0f')", grid=False
    )
)
y = alt.Y(
    'vida:Q', title='Esperanza de vida (años)',
    scale=alt.Scale(domain=[50, 88]),
    axis=alt.Axis(grid=True, gridColor='#ededed', tickCount=8)
)

linea = alt.Chart(curva).mark_line(color='#555555', strokeWidth=2).encode(x=x, y=y)
base = alt.Chart(g1).transform_filter(filtro)

puntos = base.mark_circle(size=75, stroke='white', strokeWidth=0.6).encode(
    x=x, y=y,
    color=alt.Color(
        'comparacion:N', title='Respecto del modelo',
        scale=alt.Scale(
            domain=['Por encima', 'Cerca del modelo', 'Por debajo'],
            range=['#397ba7', '#d5b84f', '#b44d53']
        ),
        legend=alt.Legend(orient='top', direction='horizontal')
    ),
    opacity=alt.condition(
        f'!({hay_busqueda}) || ({coincide})',
        alt.value(0.83), alt.value(0.13)
    ),
    tooltip=[
        alt.Tooltip('pais:N', title='País'),
        alt.Tooltip('vida:Q', title='Esperanza de vida (años)', format='.1f'),
        alt.Tooltip('esperada:Q', title='Estimación del modelo (años)', format='.1f'),
        alt.Tooltip('brecha:Q', title='Diferencia (años)', format='+.1f'),
        alt.Tooltip('pib:Q', title='PIB por habitante (PPA)', format=',.0f')
    ]
)

etiquetas = base.transform_filter(
    f'(({hay_busqueda}) && ({coincide})) || (!({hay_busqueda}) && datum.destacado)'
).mark_text(
    align='left', dx=9, dy=-8, fontSize=11,
    fontWeight='bold', color='#292929'
).encode(x=x, y=y, text='pais:N')

grafico_1 = (linea + puntos + etiquetas).add_params(
    vista, region, buscar
).properties(
    width=800, height=460,
    title={
        'text': '¿Los países con mayor ingreso viven más?',
        'subtitle': 'Esperanza de vida y PIB por habitante, 2024 · La línea muestra la estimación del modelo.'
    }
).configure_title(
    fontSize=17, subtitleFontSize=12, anchor='start'
).configure_view(stroke=None)

grafico_1


alt.LayerChart(...)

En 2024, el modelo dio cuenta del 74,4 % de la variación en esperanza de vida entre 195 economías. Costa Rica y Estados Unidos muestran cuánto pueden diferir dos países del patrón promedio. Nauru ofrece otra pista para seguir la historia: vive casi 10 años menos de lo que estima el modelo. ¿Dónde aparece cuando comparamos la longevidad con el acceso al agua?

# 4. ¿Cómo se relaciona el acceso al agua con la esperanza de vida?

Cada punto muestra la esperanza de vida observada y el porcentaje de personas que usan un servicio básico de agua para beber. El gráfico amplía el tramo de 90 % a 100 % porque ahí se concentra buena parte de los países. El color indica el grupo de ingreso. El círculo vacío en el borde inferior identifica valores por debajo de 35 años.

Mové el deslizador para comparar años o buscá un país para ver su recorrido.

In [332]:
INGRESO_ORDEN = ["Bajo", "Medio bajo", "Medio alto", "Alto", "Sin clasificar"]
INGRESO_COLORES = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b", "#b4b2a9"]
color_ingreso = alt.Color("ingreso:N", title="Ingreso del país", scale=alt.Scale(domain=INGRESO_ORDEN, range=INGRESO_COLORES),
                          legend=alt.Legend(orient="right"))
VIDA_MIN, VIDA_MAX = 35, 88

def datos_anuales(col, decimales):
    d = panel[panel["anio"].between(DESDE, HASTA) & panel[col].notna() & panel["vida"].notna()]
    return d.assign(valor=d[col].round(decimales), vida=d["vida"].round(1))[["iso3", "pais", "region", "ingreso", "anio", "valor", "vida"]]


def resumen_anual(d, cortes, etiquetas):
    grupos = pd.cut(d["valor"], cortes, labels=etiquetas, right=False)
    tabla = d.groupby(["anio", grupos], observed=False)["vida"].agg(["size", "median"]).unstack()
    tabla.columns = [f"{'países' if m == 'size' else 'mediana vida'} {g}" for m, g in tabla.columns]
    tabla.insert(0, "correlación", d.groupby("anio")[["valor", "vida"]].corr().xs("valor", level=1)["vida"])
    tabla.insert(0, "países", d.groupby("anio").size())
    return tabla.round(2)


agua = datos_anuales("agua", 1)
resumen_agua = resumen_anual(agua, [0, 80, 95, 101], ["< 80 %", "80–95 %", "≥ 95 %"])

In [333]:
def coma(v, decimales=1):
    return f"{v:.{decimales}f}".replace(".", ",")

# Texto que acompaña a cada año: cantidad de países y resumen de la relación
textos_agua = pd.DataFrame([{
    "anio": anio,
    "linea_1": f"{int(fila['países'])} países con ambos datos en {anio}",
    "linea_2": f"Correlación: {coma(fila['correlación'], 2)} · mediana de esperanza de vida: {coma(fila['mediana vida < 80 %'])} años "
               f"con menos del 80 % de acceso y {coma(fila['mediana vida ≥ 95 %'])} con 95 % o más",
} for anio, fila in resumen_agua.iterrows()])

ANCHO, ALTO = 760, 460
anio_agua = alt.param(name="anio_agua", value=HASTA, bind=alt.binding_range(min=DESDE, max=HASTA, step=1, name="Año: "))
buscar_agua = alt.param(name="buscar_agua", value="", bind=alt.binding(input="search", placeholder="Ej.: Argentina", name="Buscar un país: "))
COINCIDE_AGUA = "length(buscar_agua) > 1 && indexof(lower(datum.pais), lower(buscar_agua)) >= 0"

# Escala por tramos: de 15 % a 90 % ocupa el 55 % del ancho y de 90 % a 100 %, el 45 % restante
escala_agua = alt.Scale(domain=[15, 90, 100], range=[0, ANCHO * 0.55, ANCHO], nice=False)
eje_agua = alt.Axis(values=[20, 40, 60, 80, 90, 92, 94, 96, 98, 100], labelExpr="datum.value + ' %'")
TITULO_X_AGUA = "Población que usa al menos un servicio básico de agua para beber (%)"
escala_vida = alt.Scale(domain=[VIDA_MIN, VIDA_MAX], clamp=True)
x_agua = alt.X("valor:Q", title=TITULO_X_AGUA, scale=escala_agua, axis=eje_agua)
y_agua = alt.Y("vida:Q", title="Esperanza de vida (años)", scale=escala_vida)

tramo = alt.Chart(pd.DataFrame({"desde": [90], "hasta": [100]})).mark_rect(color="#e1e0d9", opacity=0.45).encode(
    x=alt.X("desde:Q", title=TITULO_X_AGUA, scale=escala_agua, axis=eje_agua), x2="hasta:Q")
corte_tramo = alt.Chart(pd.DataFrame({"valor": [90]})).mark_rule(strokeDash=[3, 3], color="#898781").encode(x=x_agua)
rotulo_tramo = alt.Chart(pd.DataFrame({"valor": [90.3], "vida": [VIDA_MAX - 1]})).mark_text(
    align="left", baseline="top", fontSize=11, color="#898781", text="Tramo ampliado: 90 % a 100 %").encode(x=x_agua, y=y_agua)

base_agua = alt.Chart(agua).transform_filter("datum.anio == anio_agua")
FICHA_AGUA = [
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("anio:Q", title="Año", format="d"),
        alt.Tooltip("valor:Q", title="Acceso básico a agua (% de la población)", format=".1f"),
        alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
        alt.Tooltip("ingreso:N", title="Grupo de ingreso"),
]
puntos_agua = base_agua.transform_filter(f"datum.vida >= {VIDA_MIN}").mark_circle(size=60, stroke="white", strokeWidth=0.5).encode(
    x=x_agua, y=y_agua, color=color_ingreso,
    opacity=alt.condition(f"length(buscar_agua) < 2 || {COINCIDE_AGUA}", alt.value(0.85), alt.value(0.15)),
    tooltip=FICHA_AGUA,
)
# Esperanza de vida menor de 35 años: círculo vacío sobre el borde inferior
fuera_agua = base_agua.transform_filter(f"datum.vida < {VIDA_MIN}").mark_point(size=60, filled=False, strokeWidth=1.5).encode(
    x=x_agua, y=y_agua, color=color_ingreso, tooltip=FICHA_AGUA)
# Recorrido del país buscado desde 2000 hasta el año elegido
recorrido = alt.Chart(agua).transform_filter(f"{COINCIDE_AGUA} && datum.anio <= anio_agua")
linea_recorrido = recorrido.mark_line(color="#52514e", strokeWidth=1.5, point=alt.OverlayMarkDef(size=12, color="#52514e")).encode(
    x=x_agua, y=y_agua, detail="pais:N", order="anio:Q")
nombre_recorrido = base_agua.transform_filter(COINCIDE_AGUA).mark_text(align="left", dx=8, dy=-8, fontWeight="bold").encode(
    x=x_agua, y=y_agua, text="pais:N")

encabezado_agua = alt.Chart(textos_agua).transform_filter("datum.anio == anio_agua")
linea_1_agua = encabezado_agua.mark_text(align="left", x=8, y=-28, fontSize=13, fontWeight="bold").encode(text="linea_1:N")
linea_2_agua = encabezado_agua.mark_text(align="left", x=8, y=-12, fontSize=11.5, color="#52514e").encode(text="linea_2:N")

grafico_2 = (tramo + corte_tramo + rotulo_tramo + linea_recorrido + puntos_agua + fuera_agua + nombre_recorrido + linea_1_agua + linea_2_agua).add_params(
    anio_agua, buscar_agua
).properties(
    width=ANCHO, height=ALTO,
    title={"text": "¿Viven más los países donde más gente tiene agua para beber?",
           "subtitle": ["Cada punto es un país. Las escalas quedan fijas al mover el año y solo aparecen los países con ambos datos.",
                        "Los círculos vacíos sobre el borde inferior tienen una esperanza de vida menor de 35 años. Fuente: Banco Mundial (WDI), vía Data360."],
           "offset": 40},
)
grafico_2


alt.LayerChart(...)

Entre 2000 y 2024, la diferencia de esperanza de vida mediana entre los países con menos del 80 % y los que alcanzaban al menos el 95 % de acceso bajó de unos 21 a 14,5 años. En 2024, entre los países con al menos 95 % de acceso, la esperanza de vida iba de 62 a 86 años. Nauru queda en el extremo bajo, con casi 98 % de acceso y 62 años de esperanza de vida.

El agua básica describe una parte de las condiciones de vida. El último gráfico compara la longevidad con otro indicador: el alcance de la vacunación infantil.

# 5. ¿Viven más los países que vacunan más contra el sarampión?

Arriba, cada raya representa una economía y cada rombo marca la mediana de esperanza de vida del tramo de cobertura. Abajo, las líneas siguen la mediana de cada tramo desde 2000. Los países pueden cambiar de tramo de un año a otro. Los valores por debajo de 35 años quedan en el borde izquierdo.

Mové el deslizador para elegir el año de arriba. Podés buscar un país para ver dónde cae.

In [335]:
GRUPOS = ["95 % o más", "90 % a 94 %", "80 % a 89 %", "Menos de 80 %"]      # de mayor a menor cobertura
COLORES_GRUPOS = ["#0d366b", "#1c5cab", "#3987e5", "#86b6ef"]

vacunas = datos_anuales("sarampion", 0)
vacunas["grupo"] = pd.cut(vacunas["valor"], [0, 80, 90, 95, 101], right=False, labels=GRUPOS[::-1]).astype(str)
resumen_vacunas = resumen_anual(vacunas, [0, 80, 90, 95, 101], ["< 80 %", "80–89 %", "90–94 %", "≥ 95 %"])

medianas = vacunas.groupby(["anio", "grupo"])["vida"].agg(mediana="median", n="size").reset_index()
medianas["texto"] = medianas.apply(lambda f: f"Mediana: {coma(f['mediana'])} años · {f['n']} países", axis=1)

textos_vacunas = pd.DataFrame([{
    "anio": anio,
    "linea_1": f"{int(fila['países'])} países con ambos datos en {anio}",
    "linea_2": f"Correlación entre cobertura y esperanza de vida: {coma(fila['correlación'], 2)}",
} for anio, fila in resumen_vacunas.iterrows()])

In [336]:
anio_vac = alt.param(name="anio_vac", value=HASTA, bind=alt.binding_range(min=DESDE, max=HASTA, step=1, name="Año: "))
buscar_vac = alt.param(name="buscar_vac", value="", bind=alt.binding(input="search", placeholder="Ej.: Argentina", name="Buscar un país: "))
COINCIDE_VAC = "length(buscar_vac) > 1 && indexof(lower(datum.pais), lower(buscar_vac)) >= 0"

ANCHO_3 = 640
y_grupo = alt.Y("grupo:N", title="Cobertura de vacunación", sort=GRUPOS,
                axis=alt.Axis(labelFontSize=12, labelFontWeight="bold", ticks=False, domain=False, titlePadding=12))
x_vida = alt.X("vida:Q", title="Esperanza de vida al nacer (años)", scale=escala_vida)
color_grupo = alt.Color("grupo:N", scale=alt.Scale(domain=GRUPOS, range=COLORES_GRUPOS), legend=None)

# Panel de arriba: los países del año elegido, una fila por tramo de cobertura
base_vac = alt.Chart(vacunas).transform_filter("datum.anio == anio_vac")
rayas = base_vac.mark_tick(thickness=2, size=34).encode(
    x=x_vida, y=y_grupo, color=color_grupo,
    opacity=alt.condition(f"length(buscar_vac) < 2 || {COINCIDE_VAC}", alt.value(0.6), alt.value(0.12)),
    tooltip=[
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("anio:Q", title="Año", format="d"),
        alt.Tooltip("valor:Q", title="Vacunación contra el sarampión (% de niños de 12 a 23 meses)", format="d"),
        alt.Tooltip("grupo:N", title="Tramo de cobertura"),
        alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
    ],
)
raya_buscada = base_vac.transform_filter(COINCIDE_VAC).mark_tick(thickness=3, size=44, color="#0b0b0b").encode(x=x_vida, y=y_grupo)
nombre_buscado = base_vac.transform_filter(COINCIDE_VAC).mark_text(dy=-27, fontWeight="bold", fontSize=12).encode(
    x=x_vida, y=y_grupo, text="pais:N")

base_med = alt.Chart(medianas).transform_filter("datum.anio == anio_vac")
rombos = base_med.mark_point(shape="diamond", filled=True, size=220, color="#0b0b0b", opacity=1).encode(
    x=alt.X("mediana:Q", title="Esperanza de vida al nacer (años)", scale=escala_vida), y=y_grupo,
    tooltip=[alt.Tooltip("grupo:N", title="Tramo de cobertura"), alt.Tooltip("mediana:Q", title="Mediana de esperanza de vida", format=".1f"),
             alt.Tooltip("n:Q", title="Países")])
textos_derecha = base_med.mark_text(align="left", x=ANCHO_3 + 12, fontSize=12).encode(y=y_grupo, text="texto:N")

encabezado_vac = alt.Chart(textos_vacunas).transform_filter("datum.anio == anio_vac")
linea_1_vac = encabezado_vac.mark_text(align="left", x=0, y=-30, fontSize=13, fontWeight="bold").encode(text="linea_1:N")
linea_2_vac = encabezado_vac.mark_text(align="left", x=0, y=-14, fontSize=11.5, color="#52514e").encode(text="linea_2:N")

arriba = (rayas + raya_buscada + rombos + nombre_buscado + textos_derecha + linea_1_vac + linea_2_vac).properties(
    width=ANCHO_3, height=280,
    title={"text": "¿Viven más los países que vacunan más contra el sarampión?",
           "subtitle": ["Cada raya es un país, ubicado según su esperanza de vida, en la fila de su cobertura de vacunación (primera dosis).",
                        "El rombo negro marca la mediana de cada tramo. Fuente: Banco Mundial (WDI), vía Data360."],
           "offset": 44},
)

# Panel de abajo: cómo cambió la mediana de cada tramo entre 2000 y 2024
lineas = alt.Chart(medianas).mark_line(strokeWidth=2.5, point=alt.OverlayMarkDef(size=25)).encode(
    x=alt.X("anio:Q", title="Año", axis=alt.Axis(format="d", values=list(range(DESDE, HASTA + 1, 4)))),
    y=alt.Y("mediana:Q", title="Mediana de esperanza de vida (años)", scale=alt.Scale(zero=False)),
    color=alt.Color("grupo:N", title="Cobertura de vacunación", scale=alt.Scale(domain=GRUPOS, range=COLORES_GRUPOS),
                    legend=alt.Legend(orient="right")),
    tooltip=[alt.Tooltip("grupo:N", title="Tramo de cobertura"), alt.Tooltip("anio:Q", title="Año", format="d"),
             alt.Tooltip("mediana:Q", title="Mediana de esperanza de vida", format=".1f"), alt.Tooltip("n:Q", title="Países")],
)
anio_elegido = alt.Chart(textos_vacunas).transform_filter("datum.anio == anio_vac").mark_rule(color="#0b0b0b", strokeDash=[4, 3]).encode(x="anio:Q")
abajo = (lineas + anio_elegido).properties(width=ANCHO_3, height=200, title="Mediana de esperanza de vida de cada tramo, 2000–2024")

grafico_3 = alt.vconcat(arriba, abajo, spacing=36).add_params(anio_vac, buscar_vac).resolve_scale(color="independent")
grafico_3


alt.VConcatChart(...)

En 2024, la esperanza de vida mediana fue de 66,5 años en el grupo con menos del 80 % de cobertura y de 76,8 años en el de al menos 95 %. Los tramos se superponen: Montenegro tenía casi 78 años de esperanza de vida con 23 % de cobertura de la primera dosis.

Entre 2019 y 2021, el número de economías que alcanzaban el 95 % bajó de 83 a 59. En 2024 había subido a 67.

# 6. Conclusión

El PIB por habitante describe buena parte de las diferencias de longevidad. Los otros gráficos ayudan a ver cuánto varía la esperanza de vida entre países con acceso parecido al agua o con niveles de vacunación similares. Costa Rica, Nauru y Montenegro muestran tres formas distintas de apartarse de esos patrones. Las comparaciones son entre países y muestran asociaciones. Estimar cuánto aporta cada factor a la longevidad requeriría otro diseño de análisis.

### Fuentes

Banco Mundial, *World Development Indicators*, consultados con la [API de Data360](https://data360.worldbank.org/en/api) al ejecutar el notebook. Los datos de origen son:

- **Esperanza de vida:** estimaciones de población de Naciones Unidas.
- **Acceso al agua:** Programa Conjunto de Monitoreo OMS/UNICEF (JMP).
- **Vacunación:** estimaciones OMS/UNICEF de cobertura de inmunización (WUENIC).

Las regiones y los grupos de ingreso corresponden a la clasificación vigente del Banco Mundial y se aplican a todos los años.